# Solar Flare Forecasting with NOAA GOES-18

Can recent solar X-ray measurements help estimate whether an M-class-or-stronger flare will begin in the next 60 minutes? This notebook presents the first real-data audit and chronological model evaluation. It is retrospective research, not an operational warning system. Target labels use catalogued flare **start times**; features use only measurements at or before each timestamp.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, REPORTS = ROOT / 'data', ROOT / 'reports'
features = pd.read_csv(DATA / 'processed' / 'goes18_forecast_features.csv', parse_dates=['time_utc'])
features['year'] = features['time_utc'].dt.year
print(f"{len(features):,} rows; {features['target_m_or_x_start_next_60m'].mean():.2%} positive windows")
display(features.groupby('year').agg(rows=('target_m_or_x_start_next_60m','size'), positive_rate=('target_m_or_x_start_next_60m','mean')))

## Held-out model evaluation

Training uses 2022–2023, threshold selection uses 2024, and 2025 is held out for the final evaluation. Rows whose 60-minute target window crosses a split boundary are excluded. Adjacent five-minute examples overlap, so these scores are not event-level warning skill.

In [ ]:
with (REPORTS / 'model_evaluation.json').open(encoding='utf-8') as f:
    evaluation = json.load(f)
rows=[]
for name, splits in evaluation['models'].items():
    m=splits['test_2025']
    rows.append({'Model':name,'Average precision':m['average_precision'],'ROC-AUC':m['roc_auc'],'Precision':m['precision_at_threshold'],'Recall':m['recall_at_threshold'],'F1':m['f1_at_threshold'],'Alert rate':m['alert_rate']})
display(pd.DataFrame(rows).set_index('Model').round(3))
print(f"Prevalence baseline AP: {evaluation['test_positive_rate_baseline_average_precision']:.3f}")
display(Image(filename=str(REPORTS / 'figures' / 'precision_recall_2025.png')))

## Limitations and next steps

The initial models rank positive time windows above the prevalence baseline, but the selected thresholds produce many false alerts. Next: group adjacent predictions into alert episodes, score distinct flare detection and lead time, compare simple flux-threshold baselines, and check calibration and uncertainty. Results are retrospective and must not be presented as real-time warning capability.

In [ ]:
print((REPORTS / 'initial_data_audit.md').read_text(encoding='utf-8'))
print((REPORTS / 'first_model_evaluation.md').read_text(encoding='utf-8'))